# Convo Diarization Evaluation

Evaluates LR-ASD active speaker detection accuracy against Praat speaker labels for the
`anilu_comparison` conversation recordings.

**Ground truth**: `words/word_info.csv` — Praat-diarized words with speaker labels.
Words are grouped into segments by silence gaps; the majority speaker determines whether
the patient was speaking in that segment.

**Prediction**: LR-ASD `scores.pckl` / `tracks.pckl` — per-frame active-speaker scores,
merged into a binary `speaking` flag (closest face to frame centre).

Metrics (accuracy, F1, balanced accuracy, confusion matrix) are computed **per patient
per camera** so you can pick the best-resolving angle downstream.


In [1]:
from glob import glob
from pathlib import Path

import cv2
import dill as pickle
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    f1_score, confusion_matrix, classification_report,
)
from tqdm.notebook import tqdm


## 1 — Configuration

In [ ]:
ANILU_ROOT = Path('/mnt/labworlds/Hayden/Hayden_Lab/speech_247/anilu_comparison')

# ── Ground-truth speaker labels ───────────────────────────────────────────────
# Map each patient to the Praat speaker ID that is the patient.
# Patients not listed here default to 'Speaker1'.
# Run Section 3 first to see the speaker distributions per patient.
PATIENT_SPEAKER: dict[str, str] = {
    'YFS': 1,
    'YFV': 'SPEAKER_03'
}
DEFAULT_PATIENT_SPEAKER = 'Speaker1'

# ── Praat → LR-ASD timing offset ─────────────────────────────────────────────
# `word_info.csv` onset/offset are in ms from the Praat analysis start.
# The synced video (and LR-ASD time_sec) starts at NS5 task t=0.
# For most patients these share the same origin (offset = 0).
# For patients whose Praat analysis covered a longer session, set the offset:
#   lrasd_time_s = onset_ms / 1000 - PRAAT_OFFSET_S[patient]
# Hint: look at the diagnostic cell output — if Praat onset_min is ~1800 s
# but the video is only ~1800 s long, set the offset to that onset_min value.
PRAAT_OFFSET_S: dict[str, float] = {
    # 'YFI': 1810.0,
}
DEFAULT_PRAAT_OFFSET_S = 0.0

# ── Minimum segment duration to include in evaluation ─────────────────────────
# Sentence-level segments can be very short; keep at 0 to include all,
# or raise (e.g. 1.0) to skip utterances too short for LR-ASD to be reliable.
MIN_SEG_DURATION = 0.0

# ── LR-ASD parameters ────────────────────────────────────────────────────────
FPS              = 25.0   # LR-ASD output FPS
SMOOTH_MS        = 400.0  # smoothing window in ms
LRASD_THRESHOLD  = 0.5    # min smoothed score to count frame as speaking
PROP_THRESHOLD   = 0.5    # min proportion of segment frames speaking → predict patient

print('ANILU_ROOT:', ANILU_ROOT)
print('MIN_SEG_DURATION:', MIN_SEG_DURATION)
print('LRASD_THRESHOLD:', LRASD_THRESHOLD, '  PROP_THRESHOLD:', PROP_THRESHOLD)


ANILU_ROOT: /mnt/labworlds/Hayden/Hayden_Lab/speech_247/anilu_comparison
MIN_SEG_DURATION: 0.0
LRASD_THRESHOLD: 0.5   PROP_THRESHOLD: 0.5


## 2 — LR-ASD helper functions

In [3]:
def moving_average_nan(x, window):
    x = np.asarray(x, dtype=float)
    valid = np.isfinite(x).astype(float)
    x_filled = np.nan_to_num(x, nan=0.0)
    kernel = np.ones(window, dtype=float)
    num = np.convolve(x_filled, kernel, mode='same')
    den = np.convolve(valid, kernel, mode='same')
    out = np.full_like(x, np.nan, dtype=float)
    keep = den > 0
    out[keep] = num[keep] / den[keep]
    return out


def merge_lrasd_tracks(tracks, scores, frame_width, frame_height,
                       fps=25.0, smooth_ms=400.0, threshold=0.5):
    """
    Build a per-frame DataFrame from LR-ASD tracks/scores.
    Patient heuristic: active face closest to frame centre.
    Returns columns: frame_idx, time_sec, patient_raw_score,
                     patient_smooth_score, speaker (bool)
    """
    if len(tracks) != len(scores):
        raise ValueError(f'tracks/scores mismatch: {len(tracks)} vs {len(scores)}')

    max_frame = max(
        (int(np.asarray(tr['track']['frame']).max())
         for tr in tracks if len(tr['track']['frame']) > 0),
        default=-1,
    )
    if max_frame < 0:
        raise ValueError('No frames in tracks')

    n_frames = max_frame + 1
    n_tracks = len(tracks)
    raw_mat = np.full((n_tracks, n_frames), np.nan)
    x_mat   = np.full((n_tracks, n_frames), np.nan)
    y_mat   = np.full((n_tracks, n_frames), np.nan)

    for tidx, (tr, sc) in enumerate(zip(tracks, scores)):
        frames = np.asarray(tr['track']['frame'], dtype=int)
        sc     = np.asarray(sc, dtype=float)
        xs     = np.asarray(tr['proc_track']['x'], dtype=float)
        ys     = np.asarray(tr['proc_track']['y'], dtype=float)
        n      = min(len(frames), len(sc), len(xs), len(ys))
        raw_mat[tidx, frames[:n]] = sc[:n]
        x_mat[tidx,   frames[:n]] = xs[:n]
        y_mat[tidx,   frames[:n]] = ys[:n]

    window = max(1, int(round(smooth_ms / 1000.0 * fps)))
    if window % 2 == 0:
        window += 1
    smooth_mat = np.vstack([moving_average_nan(raw_mat[i], window) for i in range(n_tracks)])

    cx, cy   = frame_width / 2.0, frame_height / 2.0
    dist_mat = np.sqrt((x_mat - cx)**2 + (y_mat - cy)**2)
    dist_mat[~np.isfinite(raw_mat)] = np.nan

    patient_raw    = np.full(n_frames, np.nan)
    patient_smooth = np.full(n_frames, np.nan)
    speaker        = np.zeros(n_frames, dtype=bool)

    for f in range(n_frames):
        active = np.where(np.isfinite(dist_mat[:, f]))[0]
        if not len(active):
            continue
        best = active[np.argmin(dist_mat[active, f])]
        patient_raw[f]    = raw_mat[best, f]
        patient_smooth[f] = smooth_mat[best, f]
        if np.isfinite(patient_smooth[f]) and patient_smooth[f] >= threshold:
            speaker[f] = True

    return pd.DataFrame({
        'frame_idx':            np.arange(n_frames, dtype=int),
        'time_sec':             np.arange(n_frames, dtype=float) / fps,
        'patient_raw_score':    patient_raw,
        'patient_smooth_score': patient_smooth,
        'speaker':              speaker,
    })


## 3 — Load ground truth from word_info.csv

Words are grouped into sentence-level segments with continuous speaker.
A new segment starts whenever **the speaker changes** or **the previous word ends with
`.` `!` `?`** (whichever comes first).
Each segment therefore has exactly one speaker, and `is_patient` is simply
`segment_speaker == PATIENT_SPEAKER`.


In [4]:
import re as _re

_TERMINAL = _re.compile(r'[.!?]["\')\]]*$')


def is_sentence_end(word: str) -> bool:
    """True if the word string ends with a sentence-terminal punctuation mark."""
    return bool(_TERMINAL.search(str(word)))


def segment_words(df: pd.DataFrame) -> pd.Series:
    """
    Assign a seg_id to each word.
    A new segment starts whenever:
      - the speaker changes, OR
      - the previous word ended with sentence-terminal punctuation (. ! ?)
    Words must be pre-sorted by onset.
    """
    seg_ids = [0]
    seg_id  = 0
    for i in range(1, len(df)):
        prev = df.iloc[i - 1]
        curr = df.iloc[i]
        speaker_changed  = curr['speaker'] != prev['speaker']
        sentence_ended   = is_sentence_end(prev['word'])
        if speaker_changed or sentence_ended:
            seg_id += 1
        seg_ids.append(seg_id)
    return pd.Series(seg_ids, index=df.index)


def load_ground_truth(anilu_root: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Returns:
      word_df  — one row per word with patient/is_patient_word/onset_s/offset_s/seg_id
      seg_df   — one row per segment with patient/seg_id/start_s/end_s/is_patient/speaker
    """
    word_rows = []
    seg_rows  = []

    for pt_dir in sorted(anilu_root.iterdir()):
        if not pt_dir.is_dir():
            continue
        pt = pt_dir.name

        for csv_name in ('word_info_timing_corrected.csv', 'word_info.csv'):
            csv_path = pt_dir / 'words' / csv_name
            if csv_path.exists():
                break
        else:
            continue

        df = pd.read_csv(csv_path)
        if 'onset' not in df.columns or 'speaker' not in df.columns:
            continue

        offset_s   = PRAAT_OFFSET_S.get(pt, DEFAULT_PRAAT_OFFSET_S)
        pt_speaker = PATIENT_SPEAKER.get(pt, DEFAULT_PATIENT_SPEAKER)

        df = df.sort_values('onset').reset_index(drop=True)
        df['onset_s']        = df['onset']  / 1000.0 - offset_s
        df['offset_s']       = df['offset'] / 1000.0 - offset_s
        df['is_patient_word'] = df['speaker'] == pt_speaker
        df['seg_id']         = segment_words(df)
        df['patient']        = pt
        word_rows.append(df)

        # Segment-level ground truth.
        # Each segment has a single continuous speaker by construction,
        # so is_patient = (segment speaker == patient speaker).
        for sid, grp in df.groupby('seg_id'):
            seg_speaker = grp['speaker'].iloc[0]
            start_s     = grp['onset_s'].min()
            end_s       = grp['offset_s'].max()
            seg_rows.append({
                'patient':    pt,
                'seg_id':     sid,
                'speaker':    seg_speaker,
                'start_s':    start_s,
                'end_s':      end_s,
                'duration_s': end_s - start_s,
                'is_patient': seg_speaker == pt_speaker,
                'n_words':    len(grp),
                'pt_speaker': pt_speaker,
                'praat_offset_s': offset_s,
            })

    word_df = pd.concat(word_rows, ignore_index=True) if word_rows else pd.DataFrame()
    seg_df  = pd.DataFrame(seg_rows)
    return word_df, seg_df


word_df, seg_df = load_ground_truth(ANILU_ROOT)
print(f'Words loaded:    {len(word_df):,}  across {word_df["patient"].nunique()} patients')
print(f'Segments total:  {len(seg_df):,}')

if not seg_df.empty:
    summary = (
        seg_df.groupby('patient')
        .agg(
            n_segs       = ('seg_id', 'count'),
            pt_speaker   = ('pt_speaker', 'first'),
            pct_patient  = ('is_patient', 'mean'),
            avg_dur_s    = ('duration_s', 'mean'),
            onset_min_s  = ('start_s', 'min'),
            onset_max_s  = ('end_s', 'max'),
        )
    )
    summary['pct_patient'] = (summary['pct_patient'] * 100).round(1)
    summary['avg_dur_s']   = summary['avg_dur_s'].round(2)
    summary['onset_min_s'] = summary['onset_min_s'].round(1)
    summary['onset_max_s'] = summary['onset_max_s'].round(1)
    print(summary)


Words loaded:    97,252  across 14 patients
Segments total:  15,203
         n_segs pt_speaker  pct_patient  avg_dur_s  onset_min_s  onset_max_s
patient                                                                     
YEY        1052   Speaker1          3.2       1.82          0.0       2347.6
YEZ        1007   Speaker1         40.0       1.63        323.5       2653.0
YFA        1497   Speaker1          7.0       1.43       1096.6       3554.6
YFC         591   Speaker1         39.8       1.70         21.4       1231.2
YFF        1254   Speaker1         28.1       1.35         18.7       1768.2
YFG         366   Speaker1         36.3       1.80         36.9        886.0
YFI         488   Speaker1         73.8       2.29       1810.6       3592.9
YFK        1189   Speaker1         44.0       3.30          2.1       4445.4
YFM        1460   Speaker1         53.8       1.27          0.0       2008.8
YFN         869   Speaker1          8.5       1.11         28.6       1146.4
YFP     

## 4 — Timing alignment diagnostic

Compare adjusted Praat onset range against LR-ASD video duration.
If they are mismatched for a patient, adjust `PRAAT_OFFSET_S` in the config cell and re-run.


In [17]:
def get_lrasd_duration(scores_path: Path, fps: float = 25.0) -> float | None:
    """Return estimated recording duration in seconds from a scores.pckl file."""
    try:
        with open(scores_path, 'rb') as f:
            scores = pickle.load(f)
        tracks_path = scores_path.parent / 'tracks.pckl'
        with open(tracks_path, 'rb') as f:
            tracks = pickle.load(f)
        if not tracks:
            return None
        max_frame = max(
            int(np.asarray(tr['track']['frame']).max())
            for tr in tracks if len(tr['track']['frame']) > 0
        )
        return (max_frame + 1) / fps
    except Exception:
        return None


print(f"{'Patient':<8} {'Praat onset min (s)':>20} {'Praat onset max (s)':>20} {'LR-ASD dur (s, 1 cam)':>22}")
print('-' * 74)
for pt in sorted(seg_df['patient'].unique()) if not seg_df.empty else []:
    pt_segs = seg_df[seg_df['patient'] == pt]
    onset_min = pt_segs['start_s'].min()
    onset_max = pt_segs['end_s'].max()
    # Pick any one camera to estimate LR-ASD duration
    sample_scores = next(iter(
        (ANILU_ROOT / pt / 'video').rglob('pywork/scores.pckl')
    ), None)
    lrasd_dur = get_lrasd_duration(sample_scores) if sample_scores else None
    dur_str = f'{lrasd_dur:.1f}' if lrasd_dur else 'n/a'
    flag = '  ← CHECK OFFSET' if lrasd_dur and abs(onset_min) > 60 else ''
    print(f'{pt:<8} {onset_min:>20.1f} {onset_max:>20.1f} {dur_str:>22}{flag}')


Patient   Praat onset min (s)  Praat onset max (s)  LR-ASD dur (s, 1 cam)
--------------------------------------------------------------------------
YEY                       0.0               2347.6                    n/a
YEZ                     323.5               2653.0                    n/a
YFA                    1096.6               3554.6                    n/a
YFC                      21.4               1231.2                 1231.1
YFF                      18.7               1768.2                 1798.0
YFG                      36.9                886.0                    n/a
YFI                    1810.6               3592.9                 1099.3  ← CHECK OFFSET
YFK                       2.1               4445.4                 4444.4
YFM                       0.0               2008.8                 4277.9
YFN                      28.6               1146.4                 1155.2
YFP                      44.2               3462.5                 3468.0
YFR                  

## 5 — Discover LR-ASD outputs

In [18]:
def discover_lrasd_outputs(anilu_root: Path) -> pd.DataFrame:
    """
    Find all completed LR-ASD runs under anilu_comparison.
    Path pattern:
      {pt}/video/{pt}/{task_id}/{cam_serial}/synced_video/{stem}/pywork/scores.pckl
    """
    rows = []
    for scores_path in sorted(anilu_root.rglob('pywork/scores.pckl')):
        # scores_path.parts: .../{pt}/video/{pt}/{task_id}/{cam}/{synced_video}/{stem}/pywork/scores.pckl
        parts = scores_path.parts
        try:
            synced_video_idx = next(i for i, p in enumerate(parts) if p == 'synced_video')
        except StopIteration:
            continue
        cam_serial = parts[synced_video_idx - 1]
        task_id    = parts[synced_video_idx - 2]
        pt         = parts[synced_video_idx - 3]
        stem       = parts[synced_video_idx + 1]        # e.g. EMU-0028_convo_18486634
        base_dir   = scores_path.parent.parent          # .../{stem}/
        pyframes   = base_dir / 'pyframes'
        tracks_path = scores_path.parent / 'tracks.pckl'
        success    = (base_dir / 'pyavi' / '_SUCCESS').exists()
        rows.append({
            'patient':     pt,
            'task_id':     task_id,
            'cam_serial':  cam_serial,
            'stem':        stem,
            'scores_path': scores_path,
            'tracks_path': tracks_path,
            'pyframes':    pyframes,
            'has_tracks':  tracks_path.exists(),
            'has_frames':  pyframes.exists() and any(pyframes.iterdir()),
            'has_success': success,
        })
    return pd.DataFrame(rows)


lrasd_meta = discover_lrasd_outputs(ANILU_ROOT)
print(f'LR-ASD outputs found: {len(lrasd_meta)}')
if not lrasd_meta.empty:
    print(f'  patients: {sorted(lrasd_meta["patient"].unique())}')
    print(f'  complete (has_tracks + has_frames): '
          f'{(lrasd_meta["has_tracks"] & lrasd_meta["has_frames"]).sum()}')
    print()
    print(lrasd_meta.groupby('patient')['cam_serial'].apply(list).rename('cameras'))


LR-ASD outputs found: 57
  patients: ['YFC', 'YFF', 'YFI', 'YFK', 'YFM', 'YFN', 'YFP', 'YFR', 'YFS', 'YFU']
  complete (has_tracks + has_frames): 57

patient
YFC             [18486634, 18486638, 23512014, 23512906]
YFF             [18486634, 18486638, 23512014, 23512906]
YFI    [18486634, 18486638, 18486644, 23505577, 23512...
YFK    [18486638, 23505577, 23512012, 23512014, 23512...
YFM    [18486638, 18486644, 23505577, 23512012, 23512...
YFN    [18486638, 18486644, 23505577, 23512012, 23512...
YFP             [23512013, 23512014, 23512906, 23512908]
YFR                       [18486638, 18486644, 23505577]
YFS    [18486638, 23505577, 23512013, 23512906, 23512...
YFU    [18486638, 18486644, 23505577, 23512012, 23512...
Name: cameras, dtype: object


## 6 — Classify segments with LR-ASD

For each (patient, camera) pair: load LR-ASD output, then for each Praat segment
check the proportion of frames where the patient is active-speaking.


In [19]:
def get_frame_dims(pyframes: Path) -> tuple[int, int] | None:
    """Return (width, height) from the first JPEG in pyframes."""
    jpgs = sorted(pyframes.glob('*.jpg'))
    if not jpgs:
        return None
    img = cv2.imread(str(jpgs[0]))
    if img is None:
        return None
    h, w = img.shape[:2]
    return w, h


def classify_segments_for_camera(
    seg_df_pt: pd.DataFrame,
    lrasd_df: pd.DataFrame,
) -> pd.DataFrame:
    """Classify each segment for one patient/camera. Returns copy of seg_df_pt with pred column."""
    result = seg_df_pt.copy()
    result['pred_patient'] = False
    result['n_frames']     = 0
    result['prop_speaking'] = np.nan

    for idx, row in result.iterrows():
        if row['duration_s'] < MIN_SEG_DURATION:
            continue
        mask = (lrasd_df['time_sec'] >= row['start_s']) & (lrasd_df['time_sec'] <= row['end_s'])
        seg_frames = lrasd_df[mask]
        if len(seg_frames) == 0:
            continue
        prop = seg_frames['speaker'].sum() / len(seg_frames)
        result.at[idx, 'n_frames']     = len(seg_frames)
        result.at[idx, 'prop_speaking'] = prop
        result.at[idx, 'pred_patient'] = prop >= PROP_THRESHOLD

    return result


all_results = []   # list of dicts with patient, cam_serial, classified seg_df
load_errors = []

valid_lrasd = lrasd_meta[lrasd_meta['has_tracks'] & lrasd_meta['has_frames']]

for _, meta_row in tqdm(valid_lrasd.iterrows(), total=len(valid_lrasd), desc='Loading LR-ASD'):
    pt         = meta_row['patient']
    cam_serial = meta_row['cam_serial']

    # Only process patients that have ground truth
    if seg_df.empty or pt not in seg_df['patient'].values:
        continue

    dims = get_frame_dims(meta_row['pyframes'])
    if dims is None:
        load_errors.append((pt, cam_serial, 'could not read frame dimensions'))
        continue
    frame_w, frame_h = dims

    try:
        with open(meta_row['scores_path'], 'rb') as f:
            scores = pickle.load(f)
        with open(meta_row['tracks_path'], 'rb') as f:
            tracks = pickle.load(f)
        lrasd_df = merge_lrasd_tracks(
            tracks=tracks, scores=scores,
            frame_width=frame_w, frame_height=frame_h,
            fps=FPS, smooth_ms=SMOOTH_MS, threshold=LRASD_THRESHOLD,
        )
    except Exception as e:
        load_errors.append((pt, cam_serial, str(e)))
        continue

    pt_segs = seg_df[seg_df['patient'] == pt].copy()
    classified = classify_segments_for_camera(pt_segs, lrasd_df)
    classified['cam_serial'] = cam_serial
    all_results.append(classified)

if load_errors:
    print(f'Load errors ({len(load_errors)}):')
    for pt, cam, err in load_errors:
        print(f'  {pt} {cam}: {err}')

if all_results:
    results_df = pd.concat(all_results, ignore_index=True)
    print(f'\nClassified {len(results_df):,} segment×camera rows')
    print(f'  ({results_df[["patient","cam_serial"]].drop_duplicates().shape[0]} patient/camera combos)')
else:
    results_df = pd.DataFrame()
    print('No results — check that LR-ASD outputs and ground truth are available.')


Loading LR-ASD:   0%|          | 0/57 [00:00<?, ?it/s]

Load errors (2):
  YFM 18486644: No frames in tracks
  YFM 23512908: No frames in tracks

Classified 62,597 segment×camera rows
  (55 patient/camera combos)


## 7 — Compute diarization metrics

Only segments with `duration_s >= MIN_SEG_DURATION` **and** at least one LR-ASD frame
(`n_frames > 0`) are included in the evaluation.


In [20]:
from sklearn.metrics import precision_score, recall_score


def compute_metrics(sub: pd.DataFrame) -> dict | None:
    """Compute classification metrics for a patient/camera subset."""
    ev = sub[(sub['duration_s'] >= MIN_SEG_DURATION) & (sub['n_frames'] > 0)]
    if len(ev) == 0:
        return None
    y_true = ev['is_patient'].astype(int)
    y_pred = ev['pred_patient'].astype(int)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm[0,0], cm[0,1], cm[1,0], cm[1,1]
    return {
        'n_segs':            len(ev),
        'n_patient':         int(y_true.sum()),
        'precision':         round(precision_score(y_true, y_pred, zero_division=0), 4),
        'recall':            round(recall_score(y_true, y_pred, zero_division=0), 4),
        'f1':                round(f1_score(y_true, y_pred, zero_division=0), 4),
        'balanced_accuracy': round(balanced_accuracy_score(y_true, y_pred), 4),
        'accuracy':          round(accuracy_score(y_true, y_pred), 4),
        'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
    }


metric_rows = []
if not results_df.empty:
    for (pt, cam), grp in results_df.groupby(['patient', 'cam_serial']):
        m = compute_metrics(grp)
        if m:
            metric_rows.append({'patient': pt, 'cam_serial': cam, **m})

metrics_df = pd.DataFrame(metric_rows)
if not metrics_df.empty:
    metrics_df = metrics_df.sort_values(
        ['patient', 'balanced_accuracy'], ascending=[True, False]
    )
    print(metrics_df[[
        'patient', 'cam_serial', 'n_segs', 'n_patient',
        'precision', 'recall', 'f1', 'balanced_accuracy', 'accuracy',
        'tn', 'fp', 'fn', 'tp',
    ]].to_string(index=False))
else:
    print('No metrics computed.')


patient cam_serial  n_segs  n_patient  precision  recall     f1  balanced_accuracy  accuracy   tn  fp  fn  tp
    YFC   18486634     591        235     0.6838  0.3957 0.5013             0.6375    0.6870  313  43 142  93
    YFC   18486638     591        235     0.0000  0.0000 0.0000             0.5000    0.6024  356   0 235   0
    YFC   23512014     591        235     0.0000  0.0000 0.0000             0.5000    0.6024  356   0 235   0
    YFC   23512906     591        235     0.0000  0.0000 0.0000             0.4902    0.5905  349   7 235   0
    YFF   18486634    1254        353     0.6598  0.3626 0.4680             0.6447    0.7679  835  66 225 128
    YFF   18486638    1254        353     0.0000  0.0000 0.0000             0.5000    0.7185  901   0 353   0
    YFF   23512014    1254        353     0.0000  0.0000 0.0000             0.5000    0.7185  901   0 353   0
    YFF   23512906    1175        328     0.1995  0.2287 0.2131             0.4366    0.5285  546 301 253  75
    YFK   

## 8 — Best camera per patient

In [21]:
if not metrics_df.empty:
    best = (
        metrics_df
        .sort_values('balanced_accuracy', ascending=False)
        .groupby('patient')
        .first()
        .reset_index()
    )
    print('Best camera per patient (by balanced accuracy):')
    print(best[[
        'patient', 'cam_serial', 'n_segs',
        'precision', 'recall', 'f1', 'balanced_accuracy', 'accuracy',
    ]].to_string(index=False))
else:
    print('No metrics to summarize.')


Best camera per patient (by balanced accuracy):
patient cam_serial  n_segs  precision  recall     f1  balanced_accuracy  accuracy
    YFC   18486634     591     0.6838  0.3957 0.5013             0.6375    0.6870
    YFF   18486634    1254     0.6598  0.3626 0.4680             0.6447    0.7679
    YFK   23512908    1189     0.6667  0.8298 0.7394             0.7520    0.7426
    YFM   23512014    1460     0.7341  0.1618 0.2651             0.5468    0.5178
    YFN   23512013     869     0.4444  0.0541 0.0964             0.5239    0.9137
    YFP   23512013     983     0.6913  0.9439 0.7981             0.7969    0.7833
    YFR   18486644     921     0.8077  0.0513 0.0966             0.5208    0.5733
    YFS   23512908    1455     0.8274  0.7852 0.8057             0.7120    0.7395
    YFU   23512013    2046     0.0000  0.0000 0.0000             0.5000    0.5806


## 9 — Per-patient confusion matrices (all cameras)

In [22]:
if not metrics_df.empty:
    for pt in sorted(metrics_df['patient'].unique()):
        pt_m = metrics_df[metrics_df['patient'] == pt].copy()
        pt_speaker = seg_df.loc[seg_df['patient'] == pt, 'pt_speaker'].iloc[0]
        print(f"\n{'='*70}")
        print(f'Patient: {pt}  (patient_speaker={pt_speaker})')
        print(f"  {'cam':<12}  {'prec':>6}  {'rec':>6}  {'f1':>6}  "
              f"{'bal_acc':>7}  {'acc':>6}  {'n':>5}  [tn  fp  fn  tp]")
        for _, row in pt_m.iterrows():
            print(f"  {row['cam_serial']:<12}  "
                  f"{row['precision']:>6.3f}  "
                  f"{row['recall']:>6.3f}  "
                  f"{row['f1']:>6.3f}  "
                  f"{row['balanced_accuracy']:>7.3f}  "
                  f"{row['accuracy']:>6.3f}  "
                  f"{row['n_segs']:>5}  "
                  f"[{row['tn']} {row['fp']} {row['fn']} {row['tp']}]")



Patient: YFC  (patient_speaker=Speaker1)
  cam             prec     rec      f1  bal_acc     acc      n  [tn  fp  fn  tp]
  18486634       0.684   0.396   0.501    0.637   0.687    591  [313 43 142 93]
  18486638       0.000   0.000   0.000    0.500   0.602    591  [356 0 235 0]
  23512014       0.000   0.000   0.000    0.500   0.602    591  [356 0 235 0]
  23512906       0.000   0.000   0.000    0.490   0.591    591  [349 7 235 0]

Patient: YFF  (patient_speaker=Speaker1)
  cam             prec     rec      f1  bal_acc     acc      n  [tn  fp  fn  tp]
  18486634       0.660   0.363   0.468    0.645   0.768   1254  [835 66 225 128]
  18486638       0.000   0.000   0.000    0.500   0.719   1254  [901 0 353 0]
  23512014       0.000   0.000   0.000    0.500   0.719   1254  [901 0 353 0]
  23512906       0.200   0.229   0.213    0.437   0.528   1175  [546 301 253 75]

Patient: YFK  (patient_speaker=Speaker1)
  cam             prec     rec      f1  bal_acc     acc      n  [tn  fp  fn  tp]

## 10 — Error analysis: false positives & false negatives

In [23]:
# Shows the worst-misclassified segments for the best camera of each patient.
if not results_df.empty and not metrics_df.empty:
    best_cams = (
        metrics_df
        .sort_values('balanced_accuracy', ascending=False)
        .groupby('patient')['cam_serial']
        .first()
        .to_dict()
    )
    for pt, cam in best_cams.items():
        sub = results_df[
            (results_df['patient'] == pt) &
            (results_df['cam_serial'] == cam) &
            (results_df['duration_s'] >= MIN_SEG_DURATION) &
            (results_df['n_frames'] > 0)
        ].copy()
        errors = sub[sub['is_patient'] != sub['pred_patient']].copy()
        errors['error_type'] = np.where(
            errors['pred_patient'], 'FP (predicted patient, was other)',
                                    'FN (missed patient speech)'
        )
        print(f'\n{pt}  cam={cam}  errors={len(errors)}/{len(sub)}')
        if len(errors) > 0:
            print(errors[['seg_id','start_s','end_s','duration_s','prop_speaking','error_type']]
                  .head(10).to_string(index=False))



YFC  cam=18486634  errors=185/591
 seg_id   start_s     end_s  duration_s  prop_speaking                 error_type
      3  28.58601  29.25014     0.66413       0.000000 FN (missed patient speech)
      5  34.77509  38.13900     3.36391       0.130952 FN (missed patient speech)
      6  38.40111  41.05181     2.65070       0.166667 FN (missed patient speech)
     21  71.14463  71.96586     0.82123       0.000000 FN (missed patient speech)
     23  72.04225  72.75300     0.71075       0.000000 FN (missed patient speech)
     25  73.05426  74.98100     1.92674       0.000000 FN (missed patient speech)
     36 100.81139 100.96993     0.15854       0.000000 FN (missed patient speech)
     44 135.76884 136.32836     0.55952       0.000000 FN (missed patient speech)
     46 138.46316 139.03239     0.56923       0.000000 FN (missed patient speech)
     49 140.90623 141.17649     0.27026       0.000000 FN (missed patient speech)

YFF  cam=18486634  errors=291/1254
 seg_id   start_s     end_s